## Smagorinsky Subgrid-Scale Heat Flux

Potential temperature $ \theta $ transported by turbulence. In LES, large scales are explicitly resolved while the effect of unresolved eddies is captured by SGF

## 1. Filtered Heat Equation and SGS Term
Applying a spatial filter (= LES) to the advection–diffusion equation for $\theta$ gives the filtered heat equation:

$\displaystyle
\frac{\partial \bar{\theta}}{\partial t}
+ \bar{u}_k \frac{\partial \bar{\theta}}{\partial x_k}
= - \frac{\partial q_k^{\mathrm{SGS}}}{\partial x_k}
+ \text{(molecular diffusion + sources)}.
$

The term $\displaystyle
q_k^{\mathrm{SGS}}
= \overline{u_k \theta} - \bar{u}_k \, \bar{\theta}$
is the (unclosed) subgrid-scale heat flux, arising from correlations between unresolved velocity and temperature fluctuations. This term must be parameterized (closed).

---

## 2. SGS Scalar Diffusivity (Smagorinsky)

In the Smagorinsky model, the SGS scalar diffusivity is $\displaystyle K_h = \frac{K_m}{Pr_T},$ where the SGS eddy viscosity is $\displaystyle K_m = (C_S \Delta)^2 |S|, $ and $\displaystyle |S| = \sqrt{2 S_{ij} S_{ij}}.$ Here:
- $C_S$ — Smagorinsky constant --> aprox. 0.2 (not defined in PALM... maybe use value of Dipankar et al. (2015)),  
- $\Delta$ — LES filter width (=effective grid width. Not sure how this is implemented in PALM, 2 options: $\Delta=(\Delta_x \Delta_y \Delta_z)^{1/3}$ or $\Delta= (\Delta_x + \Delta_y + \Delta_z)$/3... better check in code),  
- $Pr_T$ — turbulent Prandtl number,  
- $S_{ij}$ — resolved strain-rate tensor.


## 3. SGS Heat Flux Closure

The SGS heat flux is modeled as downgradient diffusion of $\bar{\theta}$: $\displaystyle q_k^{\mathrm{SGS}} = -K_h \frac{\partial \bar{\theta}}{\partial x_k}, \qquad k = 1,2,3. $

In vector form: $\displaystyle
\mathbf{q}^{\mathrm{SGS}} = - K_h \nabla \bar{\theta}.
$

## 4. Vertical SGS Heat Flux

The vertical component is

$\displaystyle
q_z^{\mathrm{SGS}} = -K_h \frac{\partial \bar{\theta}}{\partial z}.
$

Stable stratification ($\partial \bar{\theta}/\partial z > 0$) implies downward SGS heat flux.  
Unstable stratification ($\partial \bar{\theta}/\partial z < 0$) implies upward SGS heat flux.

---

## 5. Summary

The LES-filtered scalar equation contains an SGS heat flux divergence term

$\displaystyle
-\frac{\partial q_k^{\mathrm{SGS}}}{\partial x_k},
$

where the Smagorinsky SGS model provides

$\displaystyle
K_h = \frac{(C_S \Delta)^2 |S|}{Pr_T},
$

and the heat flux closure

$\displaystyle
q_k^{\mathrm{SGS}} = -K_h \, \partial_k \bar{\theta}.
$

The vertical SGS heat flux is therefore

$\displaystyle
q_z^{\mathrm{SGS}} = -K_h \, \partial_z \bar{\theta}.
$

These relations form the standard LES SGS closure for scalar transport using the Smagorinsky model.


## 6. Simple example

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Helper: cyclic (periodic) central gradient in x or y
# ============================================================
def grad_cyclic(f, dx, axis):
    """
    Central difference with periodic BC:
    df/dx ≈ (f_{i+1} - f_{i-1}) / (2 dx)
    using np.roll for cyclic wrap.
    """
    return (np.roll(f, -1, axis=axis) - np.roll(f, 1, axis=axis)) / (2.0 * dx)

In [ ]:
# ============================================================
# Grid setup
# ============================================================
nz, ny, nx = 10, 4, 4
dx = 2000.0   # m
dy = 2000.0   # m
dz = 100.0     # m

x = np.arange(nx) * dx
y = np.arange(ny) * dy
z = np.arange(nz) * dz

In [ ]:
# ============================================================
# Define a non-idealized 3D velocity field (u, v, w)
# (Test to compute Smagorinsky Km)
# ============================================================
u = np.zeros((nz, ny, nx))
v = np.zeros((nz, ny, nx))
w = np.zeros((nz, ny, nx))

U0 = 5.0        # base wind [m/s]
dU_dz = 0.005   # vertical shear [1/s]

for k in range(nz):
    for j in range(ny):
        for i in range(nx):
            # Shear in z + small horizontal variability
            u[k, j, i] = U0 + dU_dz*z[k] + 0.2*(i - nx/2) + 0.1*(j - ny/2)
            # Weak horizontal structure in v
            v[k, j, i] = 0.5 + 0.03 * (x[i]/dx) + 0.02*(j - ny/2)
            # Small structured vertical velocity
            w[k, j, i] = 0.05 * np.sin(z[k]/200.0) * (1 + 0.2*i + 0.1*j)

The estimation of the strain-rate tensor is closely linked to the grid (Arakawa-C). 
You basically need gradients of all velocities in the directions. Problem is that all velocities and corresponding gradients are determined at different locations. If you want to be precise, you have to look into the code. 

In [ ]:
# ============================================================
# Velocity gradients with cyclic BC in x,y and standard in z
# ============================================================
# Horizontal derivatives: periodic
du_dx = grad_cyclic(u, dx, axis=2)
du_dy = grad_cyclic(u, dy, axis=1)
dv_dx = grad_cyclic(v, dx, axis=2)
dv_dy = grad_cyclic(v, dy, axis=1)
dw_dx = grad_cyclic(w, dx, axis=2)
dw_dy = grad_cyclic(w, dy, axis=1)

# Vertical derivatives: regular central differences (non-cyclic)
du_dz = np.zeros_like(u)
dv_dz = np.zeros_like(v)
dw_dz = np.zeros_like(w)

du_dz[1:-1, :, :] = (u[2:, :, :] - u[:-2, :, :]) / (2.0 * dz)
dv_dz[1:-1, :, :] = (v[2:, :, :] - v[:-2, :, :]) / (2.0 * dz)
dw_dz[1:-1, :, :] = (w[2:, :, :] - w[:-2, :, :]) / (2.0 * dz)

# Simple one-sided at top/bottom for completeness (not crucial here)
du_dz[0, :, :]  = (u[1, :, :] - u[0, :, :]) / dz
du_dz[-1, :, :] = (u[-1, :, :] - u[-2, :, :]) / dz
dv_dz[0, :, :]  = (v[1, :, :] - v[0, :, :]) / dz
dv_dz[-1, :, :] = (v[-1, :, :] - v[-2, :, :]) / dz
dw_dz[0, :, :]  = (w[1, :, :] - w[0, :, :]) / dz
dw_dz[-1, :, :] = (w[-1, :, :] - w[-2, :, :]) / dz

# ============================================================
# Strain-rate tensor S_ij (x,y,z ~ 1,2,3)
# ============================================================
S_xx = du_dx
S_yy = dv_dy
S_zz = dw_dz

S_xy = 0.5 * (du_dy + dv_dx)
S_xz = 0.5 * (du_dz + dw_dx)
S_yz = 0.5 * (dv_dz + dw_dy)

# ============================================================
# Strain magnitude |S| = sqrt(2 S_ij S_ij)
# ============================================================
S2 = 2.0 * (
    S_xx**2 + S_yy**2 + S_zz**2 +
    2.0*(S_xy**2 + S_xz**2 + S_yz**2)
)
S_mag = np.sqrt(S2)

In [ ]:
# ============================================================
# Smagorinsky eddy viscosity: Km = nu_T = (C_s * Δ)^2 |S|
# ============================================================
C_s = 0.2
Delta = (dx * dy * dz) ** (1/3)   # cubic root of cell volume
Km = (C_s * Delta)**2 * S_mag     # momentum diffusivity

In [ ]:
print(np.shape(Delta))
print(np.shape(Km))
print(np.shape(S_mag))

In [ ]:
# ============================================================
# Define a scalar field theta (e.g. potential temperature)
# with a lapse rate + inversion + horizontal structure
# ============================================================
theta = np.zeros((nz, ny, nx))

for k in range(nz):
    for j in range(ny):
        for i in range(nx):
            # Base vertical profile
            if z[k] < 500.0:
                base = 300.0 + 0.004 * z[k]          # -4 K/km
            else:
                base = 302.0 + 0.010 * (z[k] - 500)  # +10 K/km inversion
            # Add weak horizontal variations
            theta[k, j, i] = base + 0.05*(i - nx/2) + 0.03*(j - ny/2)

In [ ]:
# ============================================================
# Scalar gradients ∇ theta with cyclic BC in x,y
# ============================================================
dtheta_dx = grad_cyclic(theta, dx, axis=2)
dtheta_dy = grad_cyclic(theta, dy, axis=1)

# Vertical derivative: central diff, bottom/top not used for flux
dtheta_dz = np.zeros_like(theta)
dtheta_dz[1:-1, :, :] = (theta[2:, :, :] - theta[:-2, :, :]) / (2.0 * dz)

# Ignore the surface and top fluxes??
dtheta_dz[0, :, :]  = 0.0
dtheta_dz[-1, :, :] = 0.0

In [ ]:
# ============================================================
# SGS heat flux (Smagorinsky):
#   Kh = Km / Pr_T   (turbulent Prandtl number)
#   F_theta_k = - Kh * ∂theta/∂x_k
# Vertical flux: Ftheta_z
# ============================================================
Pr_T = 0.7
Kh = Km / Pr_T

Ftheta_x = -Kh * dtheta_dx
Ftheta_y = -Kh * dtheta_dy
Ftheta_z = -Kh * dtheta_dz  # vertical SGS heat flux

# Enforce: neglect bottom and top fluxes, mark as missing
Ftheta_z[0, :, :]  = np.nan
Ftheta_z[-1, :, :] = np.nan

In [ ]:
# ============================================================
# Inspect vertical profile at one column (y=0, x=0)
# ============================================================
j0, i0 = 0, 0

df = pd.DataFrame({
    "z [m]"          : z,
    "theta [K]"      : theta[:, j0, i0],
    "dθ/dz [K/m]"    : dtheta_dz[:, j0, i0],
    "Km [m2/s]"      : Km[:, j0, i0],
    "Kh [m2/s]"      : Kh[:, j0, i0],
    "Fθ_z [K·m/s]"   : Ftheta_z[:, j0, i0],
})
print(df)

In [ ]:
# ============================================================
# Plot vertical SGS heat flux vs height
# ============================================================
plt.figure(figsize=(5, 6))
plt.plot(Ftheta_z[:, j0, i0], z[:], marker="o")
plt.xlabel(r"$F_{\theta,z}$ [K·m/s]")
plt.ylabel("z [m]")
plt.title("Smagorinsky SGS vertical heat flux\n(column y=0, x=0)")
plt.grid(True)
plt.gca().invert_yaxis()
plt.show()

## 7. Our coarse-grained data

In [ ]:
import pandas as pd
data_path = 'QML2_a1_cg200_10z.txt'
df = pd.read_csv(data_path).dropna()

In [ ]:
dx = 2000; dy = 2000; dz = 100

In [ ]:
x_unique = np.sort(df['X'].unique())
y_unique = np.sort(df['Y'].unique())
z_unique = np.sort(df['Z'].unique())

u = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
v = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
w = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
theta = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)

#select time step
df1 = df[df.time == df.time.iloc[100]]

for iz, z in enumerate(z_unique):
    pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='u_cg')
    pivot = pivot.reindex(index=y_unique, columns=x_unique)
    u[iz, :, :] = pivot.values

    pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='v_cg')
    pivot = pivot.reindex(index=y_unique, columns=x_unique)
    v[iz, :, :] = pivot.values

    pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='w_cg')
    pivot = pivot.reindex(index=y_unique, columns=x_unique)
    w[iz, :, :] = pivot.values

    pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='theta')
    pivot = pivot.reindex(index=y_unique, columns=x_unique)
    theta[iz, :, :] = pivot.values


In [ ]:
# ============================================================
# Velocity gradients with cyclic BC in x,y and standard in z
# ============================================================
# Horizontal derivatives: periodic
du_dx = grad_cyclic(u, dx, axis=2)
du_dy = grad_cyclic(u, dy, axis=1)
dv_dx = grad_cyclic(v, dx, axis=2)
dv_dy = grad_cyclic(v, dy, axis=1)
dw_dx = grad_cyclic(w, dx, axis=2)
dw_dy = grad_cyclic(w, dy, axis=1)

# Vertical derivatives: regular central differences (non-cyclic)
du_dz = np.zeros_like(u)
dv_dz = np.zeros_like(v)
dw_dz = np.zeros_like(w)

du_dz[1:-1, :, :] = (u[2:, :, :] - u[:-2, :, :]) / (2.0 * dz)
dv_dz[1:-1, :, :] = (v[2:, :, :] - v[:-2, :, :]) / (2.0 * dz)
dw_dz[1:-1, :, :] = (w[2:, :, :] - w[:-2, :, :]) / (2.0 * dz)

# Simple one-sided at top/bottom for completeness (not crucial here)
du_dz[0, :, :]  = (u[1, :, :] - u[0, :, :]) / dz
du_dz[-1, :, :] = (u[-1, :, :] - u[-2, :, :]) / dz
dv_dz[0, :, :]  = (v[1, :, :] - v[0, :, :]) / dz
dv_dz[-1, :, :] = (v[-1, :, :] - v[-2, :, :]) / dz
dw_dz[0, :, :]  = (w[1, :, :] - w[0, :, :]) / dz
dw_dz[-1, :, :] = (w[-1, :, :] - w[-2, :, :]) / dz

# ============================================================
# Strain-rate tensor S_ij (x,y,z ~ 1,2,3)
# ============================================================
S_xx = du_dx
S_yy = dv_dy
S_zz = dw_dz

S_xy = 0.5 * (du_dy + dv_dx)
S_xz = 0.5 * (du_dz + dw_dx)
S_yz = 0.5 * (dv_dz + dw_dy)

# ============================================================
# Strain magnitude |S| = sqrt(2 S_ij S_ij)
# ============================================================
S2 = 2.0 * (
    S_xx**2 + S_yy**2 + S_zz**2 +
    2.0*(S_xy**2 + S_xz**2 + S_yz**2)
)
S_mag = np.sqrt(S2)

In [ ]:
# ============================================================
# Smagorinsky eddy viscosity: Km = nu_T = (C_s * Δ)^2 |S|
# ============================================================
C_s = 0.2
Delta = (dx * dy * dz) ** (1/3)   # cubic root of cell volume
Km = (C_s * Delta)**2 * S_mag     # momentum diffusivity

In [ ]:
# ============================================================
# Scalar gradients ∇ theta with cyclic BC in x,y
# ============================================================
dtheta_dx = grad_cyclic(theta, dx, axis=2)
dtheta_dy = grad_cyclic(theta, dy, axis=1)

# Vertical derivative: central diff, bottom/top not used for flux
dtheta_dz = np.zeros_like(theta)
dtheta_dz[1:-1, :, :] = (theta[2:, :, :] - theta[:-2, :, :]) / (2.0 * dz)

# Ignore the surface and top fluxes??
dtheta_dz[0, :, :]  = 0.0
dtheta_dz[-1, :, :] = 0.0

In [ ]:
# ============================================================
# SGS heat flux (Smagorinsky):
#   Kh = Km / Pr_T   (turbulent Prandtl number)
#   F_theta_k = - Kh * ∂theta/∂x_k
# Vertical flux: Ftheta_z
# ============================================================
Pr_T = 0.7
Kh = Km / Pr_T

Ftheta_x = -Kh * dtheta_dx
Ftheta_y = -Kh * dtheta_dy
Ftheta_z = -Kh * dtheta_dz  # vertical SGS heat flux

# Enforce: neglect bottom and top fluxes, mark as missing
Ftheta_z[0, :, :]  = np.nan
Ftheta_z[-1, :, :] = np.nan

In [ ]:
plt.figure(figsize=(5, 6))
# ============================================================
# Inspect vertical profile at one column (y=0, x=0)
# ============================================================
for j0 in range(4):
    for i0 in range(4): 
        plt.plot(Ftheta_z[:, j0, i0], z_unique*100, marker="o")
plt.xlabel(r"$F_{\theta,z}$ [K·m/s]")
plt.ylabel("z [m]")
plt.title("Smagorinsky SGS vertical heat flux\n(column y=0, x=0)")
plt.grid(True)
#plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(5, 6))
tt = 100
for j0 in range(1):
    for i0 in range(1): 
        plt.plot(Ftheta_z[:, j0, i0], z_unique*100, marker="o")
plt.plot(-df[(df.time == df.time.iloc[tt]) & (df.X == 0) & (df.Y == 0)].theta_w_SGS, df[(df.time == df.time.iloc[tt]) & (df.X == 0) & (df.Y == 0)].Z*100,'k.-')
plt.xlabel(r"$F_{\theta,z}$ [K·m/s]")
plt.ylabel("z [m]")
plt.title("Smagorinsky SGS vertical heat flux")
plt.grid(True)
#plt.ylim(200, 800)
#plt.xlim(-0.1, 0.5)

### function to iterate over the whole data set

In [ ]:
dx = 2000; dy = 2000; dz = 100
        
# df will be dataframe of each palm run seperately
# tt iterates of iloc in time
def calc_Smag(df, tt):
        x_unique = np.sort(df['X'].unique())
        y_unique = np.sort(df['Y'].unique())
        z_unique = np.sort(df['Z'].unique())
        
        u = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
        v = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
        w = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
        theta = np.full((len(z_unique), len(y_unique), len(x_unique)), np.nan)
        
        df1 = df[df.time == df.time.iloc[tt]]
        
        for iz, z in enumerate(z_unique):
            pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='u_cg')
            pivot = pivot.reindex(index=y_unique, columns=x_unique)
            u[iz, :, :] = pivot.values
        
            pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='v_cg')
            pivot = pivot.reindex(index=y_unique, columns=x_unique)
            v[iz, :, :] = pivot.values
        
            pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='w_cg')
            pivot = pivot.reindex(index=y_unique, columns=x_unique)
            w[iz, :, :] = pivot.values
        
            pivot = df1[df1['Z'] == z].pivot(index='Y', columns='X', values='theta')
            pivot = pivot.reindex(index=y_unique, columns=x_unique)
            theta[iz, :, :] = pivot.values
        
        
        # ============================================================
        # Velocity gradients with cyclic BC in x,y and standard in z
        # ============================================================
        # Horizontal derivatives: periodic
        du_dx = grad_cyclic(u, dx, axis=2)
        du_dy = grad_cyclic(u, dy, axis=1)
        dv_dx = grad_cyclic(v, dx, axis=2)
        dv_dy = grad_cyclic(v, dy, axis=1)
        dw_dx = grad_cyclic(w, dx, axis=2)
        dw_dy = grad_cyclic(w, dy, axis=1)
        
        # Vertical derivatives: regular central differences (non-cyclic)
        du_dz = np.zeros_like(u)
        dv_dz = np.zeros_like(v)
        dw_dz = np.zeros_like(w)
        
        du_dz[1:-1, :, :] = (u[2:, :, :] - u[:-2, :, :]) / (2.0 * dz)
        dv_dz[1:-1, :, :] = (v[2:, :, :] - v[:-2, :, :]) / (2.0 * dz)
        dw_dz[1:-1, :, :] = (w[2:, :, :] - w[:-2, :, :]) / (2.0 * dz)
        
        # Simple one-sided at top/bottom for completeness (not crucial here)
        du_dz[0, :, :]  = (u[1, :, :] - u[0, :, :]) / dz
        du_dz[-1, :, :] = (u[-1, :, :] - u[-2, :, :]) / dz
        dv_dz[0, :, :]  = (v[1, :, :] - v[0, :, :]) / dz
        dv_dz[-1, :, :] = (v[-1, :, :] - v[-2, :, :]) / dz
        dw_dz[0, :, :]  = (w[1, :, :] - w[0, :, :]) / dz
        dw_dz[-1, :, :] = (w[-1, :, :] - w[-2, :, :]) / dz
        
        # ============================================================
        # Strain-rate tensor S_ij (x,y,z ~ 1,2,3)
        # ============================================================
        S_xx = du_dx
        S_yy = dv_dy
        S_zz = dw_dz
        
        S_xy = 0.5 * (du_dy + dv_dx)
        S_xz = 0.5 * (du_dz + dw_dx)
        S_yz = 0.5 * (dv_dz + dw_dy)
        
        # ============================================================
        # Strain magnitude |S| = sqrt(2 S_ij S_ij)
        # ============================================================
        S2 = 2.0 * (
            S_xx**2 + S_yy**2 + S_zz**2 +
            2.0*(S_xy**2 + S_xz**2 + S_yz**2)
        )
        S_mag = np.sqrt(S2)
        
        # ============================================================
        # Smagorinsky eddy viscosity: Km = nu_T = (C_s * Δ)^2 |S|
        # ============================================================
        C_s = 0.2
        Delta = (dx * dy * dz) ** (1/3)   # cubic root of cell volume
        Km = (C_s * Delta)**2 * S_mag     # momentum diffusivity
        
        # ============================================================
        # Scalar gradients ∇ theta with cyclic BC in x,y
        # ============================================================
        dtheta_dx = grad_cyclic(theta, dx, axis=2)
        dtheta_dy = grad_cyclic(theta, dy, axis=1)
        
        dtheta_dz = np.zeros_like(theta)
        dtheta_dz[1:-1, :, :] = (theta[2:, :, :] - theta[:-2, :, :]) / (2.0 * dz)
        
        dtheta_dz[0, :, :]  = 0.0
        dtheta_dz[-1, :, :] = 0.0
        
        # ============================================================
        # SGS heat flux (Smagorinsky):
        #   Kh = Km / Pr_T   (turbulent Prandtl number)
        #   F_theta_k = - Kh * ∂theta/∂x_k
        # Vertical flux: Ftheta_z
        # ============================================================
        Pr_T = 0.7
        Kh = Km / Pr_T
        
        Ftheta_x = -Kh * dtheta_dx
        Ftheta_y = -Kh * dtheta_dy
        Ftheta_z = -Kh * dtheta_dz  # vertical SGS heat flux
        
        # bottom and top slice are nan
        Ftheta_z[0, :, :]  = np.nan
        Ftheta_z[-1, :, :] = np.nan

        t_val = df.time.iloc[tt]
        
        Z, Y, X = np.meshgrid(z_unique, y_unique, x_unique, indexing="ij")
        
        df_Ftheta = pd.DataFrame({
            "time": t_val,
            "X": X.ravel(),
            "Y": Y.ravel(),
            "Z": Z.ravel(),
            "Ftheta_z": Ftheta_z.ravel()
        })
        
        return df_Ftheta

In [ ]:
DATA_PATH_TEMPLATE = "/work/bd1179/b309252/QML2/QML2_{}_cg200_10z.txt"
SERIES = [f"{s}{i}" for s in "abc" for i in range(1, 7)]
from scipy.spatial import cKDTree

df_all = pd.DataFrame()
for ser in SERIES[:1]:
    df = pd.read_csv(DATA_PATH_TEMPLATE.format(ser)).dropna()
    dfT = pd.DataFrame()
    for tt in range(0,len(df.time.drop_duplicates())):
        df_Ftheta = calc_Smag(df, tt)
        dfT = pd.concat([dfT, df_Ftheta])
    

    # select the matching coordinates with cKDTree
    tree = cKDTree(dfT[['time','X','Y','Z']].values)
    
    # closest match
    dist, idx = tree.query(df[['time','X','Y','Z']].values, k=1)
    
    df['Ftheta_z'] = dfT.iloc[idx]['Ftheta_z'].values
    
    df_all = pd.concat([df_all, df])

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score( -dfx.theta_w_SGS, dfx.Ftheta_z)

In [ ]:
dfx = df_all[(df_all.zrel_u < 0.8) & (df_all.zrel_u > 0.2)]
dfx = dfx[dfx.time == dfx.time.iloc[1]]
plt.plot(dfx.Ftheta_z, dfx.Z*100,'. ')
plt.plot(-dfx.theta_w_SGS, dfx.Z*100,'. ')